# ADS v2: shared working tables (sales

Builds the shared working tables: `policies_sold`, `relay_document_event`, `renewal_cycles`, `renewal_scv_quotes`, `hfq_priced`, `hfq_payment`, `contact_link`, `call_attribution`, `scv_lookups`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("10_Common", "started")


## Shared: policies_sold

Active sales before the effective date (notebook 04 cell 17, stg.GlobalPoliciesSold). Every journey reads it.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_policies_sold                                                               */
/*  Description : Active sales (Cat A1, OrgID 1, SaleCount 1) before the effective date.                */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicySale                                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_policies_sold as
select  PolicyId, PolicyCode, Channel, ReportingSaleType, PolicyTypeGroup, QuoteQueryGuid, HEQReference, ReportingSaleDate
from    ${ads.ods}.PolicySale
where   EffectiveDate < date '${ads.effective_date}'
and     ReportingSaleCategory = 'Cat A1: Active Sale'
and     OrgID = 1
and     SaleCount = 1

## Shared: relay_document_event

Relay document events from a month before the window to a month after it (families apply their own window), as notebooks 05 / 10 built ods.EventStream source 3: the description is 'business process - event type - document' (null when the event type has no name, as before), the policy's product from Policy (Motor / Home / Van / Other), and cash receipts excluded (change of 2026-07-23).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_relay_document_event                                                        */
/*  Description : Relay document events per policy around the window, with the old EventStream descript */
/*              : ion and product.                                                                      */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyDocumentEvent, ods_v2.Policy                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_relay_document_event as
select  d.EventId,
        d.DocumentId,
        d.PolicyCode,
        d.PolicyId,
        p.PolicyTypeGroup,
        d.EventDate                                                     as EventDateTime,
        concat(d.BusinessProcess, ' - ', d.EventType, ' - ', d.DocumentDescription) as EventDescription
from    ${ads.ods}.PolicyDocumentEvent d
join    ${ads.ods}.Policy p     on p.PolicyCode = d.PolicyCode
where   p.PolicyTypeGroup in ('Motor', 'Home', 'Van', 'Other')
and     d.EventDate >= add_months(timestamp '${ads.start_time}', -1)
and     d.EventDate <  add_months(timestamp '${ads.end_time}', 1)
and     coalesce(concat(d.BusinessProcess, ' - ', d.EventType, ' - ', d.DocumentDescription), '')
            not like 'Unknown - Saved Document - Cash Receipt No: %'

## Shared: renewal_cycles

Renewal cycles for every product (D-E1.19, D-E1.21, D-E1.26): the finance team's cycles (PolicyRenewal, PolicyOfferNum = 1, renew date from the window start) with their PolicyOfferNum / PolicyRetNum, plus the operations team's +/-30-day rule on top: a cycle is also renewed when the same customer bought a policy of the same product starting within 30 days either side of the end date (the segmentation V3 matching, one-to-one, best evidence first: same code, warehouse lineage, same vehicle / address, date only). The resolved renewal (finance retained, else the +/-30-day match) gives the renewal date, policy and channel; online = renewal channel 'e) Online' or a portal payment, else call centre. Lapsed = no renewal by the due date; renewed inside the window = renewed after the due date and within due + 7 days online or due + 30 days call centre; lost = not renewed within that window. Outcomes are final once due + 30 days is before the window end.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_terms                                                               */
/*  Description : The finance renewal cycles with the customer (SCV key by client code), address and ve */
/*              : hicle keys, and the end date (cancellation inside the term, else the due date).       */
/*              :                                                                                       */
/*              : Reads: ods_v2.PolicyRenewal, ods_v2.Policy, ods.scveadmlinkfinal, ods_v2.SourceCustom */
/*              : er                                                                                    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_terms as
select  r.PolicyCode,
        r.LYPolicyTypeGroup,
        r.LYPolicyRenewDateAdj                                          as RenewalDate,
        r.TYPolicyCode,
        r.TYPolicyRenewDateAdj,
        r.PolicyOfferNum,
        r.PolicyRetNum,
        r.TYReportingSaleDate,
        r.RenEURPremOffer,
        r.RenEURPremInvite,
        r.RenEURFee,
        r.TYEURFees,
        r.TYEURGrossPremium,
        r.Channel,
        r.RenewalMonth,
        r.LYVehicleRegKey                                               as VehicleRegKey,
        p.ClientCode,
        c.SCV_Customer_Key,
        a.AddressKey,
        case when cast(p.PolicyCancelDate as date) >  add_months(r.LYPolicyRenewDateAdj, -12)
              and cast(p.PolicyCancelDate as date) <  r.LYPolicyRenewDateAdj
             then cast(p.PolicyCancelDate as date) end                  as CancelDate
from    ${ads.ods}.PolicyRenewal r
left join ${ads.ods}.Policy p   on p.PolicyCode = r.PolicyCode
left join (
        select  upper(trim(ChillSourceCustomerReference)) as ClientCode, min(SCV_Customer_Key) as SCV_Customer_Key
        from    ${ads.link}.scveadmlinkfinal
        where   SourceSystemId = 1 and ChillSourceCustomerReference is not null
        group by upper(trim(ChillSourceCustomerReference))
        ) c     on c.ClientCode = p.ClientCode
left join (
        select  upper(trim(SourceSystemCustomerId)) as ClientCode, min(AddressKey) as AddressKey
        from    ${ads.ods}.SourceCustomer
        where   SourceSystemId = 1
        group by upper(trim(SourceSystemCustomerId))
        ) a     on a.ClientCode = p.ClientCode
where   r.PolicyOfferNum = 1
and     r.LYPolicyRenewDateAdj >= date '${ads.start_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_pairs                                                               */
/*  Description : Candidate +/-30-day renewals: each cycle x the same customer's active sales of the sa */
/*              : me product whose term starts within 30 days of the cycle's end date, with the evidenc */
/*              : e tier (1 same code, 2 warehouse lineage, 3 same vehicle / address, 4 date only). Tem */
/*              : porary and NB Not Taken Up new codes never count; a renewal sale naming another prede */
/*              : cessor matches only that predecessor.                                                 */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_terms, ods_v2.Policy, ods_v2.PolicySale, ods.scveadmlinkfina */
/*              : l, ods_v2.SourceCustomer                                                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_pairs as
select  t.LYPolicyTypeGroup                                             as product,
        t.SCV_Customer_Key,
        t.PolicyCode                                                    as due_code,
        t.RenewalDate                                                   as due_date,
        s.PolicyCode                                                    as cand_code,
        concat(s.PolicyCode, '|', cast(s.ReportingSaleDate as string))  as cand_key,
        s.ReportingSaleDate                                             as cand_sale_date,
        s.Channel                                                       as cand_channel,
        case when s.PolicyCode = t.PolicyCode                                                        then 1
             when s.LYPolicyCode = t.PolicyCode or s.PolicyCode = t.TYPolicyCode                     then 2
             when t.LYPolicyTypeGroup in ('Motor', 'Van') and s.VehicleRegKey is not null
                  and s.VehicleRegKey = t.VehicleRegKey                                             then 3
             when t.LYPolicyTypeGroup = 'Home' and sa.AddressKey is not null and sa.AddressKey = t.AddressKey then 3
             else 4 end                                                 as tier,
        abs(datediff(add_months(s.PolicyRenewDateAdj, -12), coalesce(t.CancelDate, t.RenewalDate))) as gap_days,
        case when t.CancelDate is null or t.CancelDate >= date_sub(t.RenewalDate, 30) then 1 else 2 end as link_rank
from    ${ads.stg}.adsv2_renewal_terms t
join    ${ads.ods}.Policy sp            on sp.PolicyTypeGroup = t.LYPolicyTypeGroup
join    ${ads.ods}.PolicySale s         on s.PolicyCode = sp.PolicyCode
join    (
        select  upper(trim(ChillSourceCustomerReference)) as ClientCode, min(SCV_Customer_Key) as SCV_Customer_Key
        from    ${ads.link}.scveadmlinkfinal
        where   SourceSystemId = 1 and ChillSourceCustomerReference is not null
        group by upper(trim(ChillSourceCustomerReference))
        ) sc    on sc.ClientCode = sp.ClientCode and sc.SCV_Customer_Key = t.SCV_Customer_Key
left join (
        select  upper(trim(SourceSystemCustomerId)) as ClientCode, min(AddressKey) as AddressKey
        from    ${ads.ods}.SourceCustomer
        where   SourceSystemId = 1
        group by upper(trim(SourceSystemCustomerId))
        ) sa    on sa.ClientCode = sp.ClientCode
where   coalesce(s.SaleCount, 0) >= 1
and     s.PolicyRenewDateAdj is not null
and     add_months(s.PolicyRenewDateAdj, -12) between date_sub(coalesce(t.CancelDate, t.RenewalDate), 30)
                                                  and date_add(coalesce(t.CancelDate, t.RenewalDate), 30)
and     (s.PolicyCode = t.PolicyCode
         or (not (s.ReportingSaleType = 'Renewals' and s.LYPolicyCode is not null and s.LYPolicyCode <> t.PolicyCode)
             and coalesce(sp.PolicyStatusDesc, '') not in ('Temporary', 'NB Not Taken Up')))

In [ ]:
%%pyspark
# stg.adsv2_renewal_match: One-to-one +/-30-day matching per customer and product (segmentation V3); renewals before rebooks, best tier, nearest start, codes break ties.
#
# Reads: stg.adsv2_renewal_pairs
import pandas as pd
pairs = spark.table(f"{stg_schema}.adsv2_renewal_pairs")

def one_to_one(pdf):
    pdf = pdf.sort_values(["link_rank", "tier", "gap_days", "due_code", "cand_key"]).reset_index(drop=True)
    due_keys = pdf["due_code"] + "|" + pdf["due_date"].astype(str)
    used_due, used_cand, keep = set(), set(), []
    for i, (d, c) in enumerate(zip(due_keys, pdf["cand_key"])):
        if d in used_due or c in used_cand:
            continue
        used_due.add(d)
        used_cand.add(c)
        keep.append(i)
    return pdf.iloc[keep]

def per_partition(batches):
    parts = list(batches)
    if not parts:
        return
    pdf = pd.concat(parts, ignore_index=True)
    out = [one_to_one(g) for _, g in pdf.groupby(["product", "SCV_Customer_Key"], sort=False, dropna=False)]
    if out:
        res = pd.concat(out, ignore_index=True)
        for c in res.columns:                       # plain object strings: newer pandas uses Arrow-backed
            if res[c].dtype.name in ("str", "string"):  # strings, which Spark 3.5's mapInPandas rejects
                res[c] = res[c].astype(object)
        yield res

(pairs.repartition(200, "product", "SCV_Customer_Key")
      .mapInPandas(per_partition, schema=pairs.schema)
      .write.mode("overwrite").format("delta").option("overwriteSchema", "true")
      .saveAsTable(f"{stg_schema}.adsv2_renewal_match"))


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_base                                                                */
/*  Description : One row per renewal cycle: the finance flags, the +/-30-day match, the resolved renew */
/*              : al (date, policy, channel, online / call centre) and the outcome flags (lapsed, renew */
/*              : ed inside the window, lost, final).                                                   */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_terms, stg.adsv2_renewal_match, ods_v2.RenewalPortalEvent    */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_base as
select  x.*,
        case when x.RenewedDate is null or x.RenewedDate > x.RenewalDate then 1 else 0 end      as Lapsed,
        case when x.RenewedDate > x.RenewalDate
              and x.RenewedDate <= date_add(x.RenewalDate, case when x.RenewalMode = 'online' then 7 else 30 end)
             then 1 else 0 end                                                                as RenewedInWindow,
        case when x.RenewedDate is null
               or x.RenewedDate > date_add(x.RenewalDate, case when x.RenewalMode = 'online' then 7 else 30 end)
             then 1 else 0 end                                                                as Lost,
        case when date_add(x.RenewalDate, 30) < date '${ads.end_time}' then 1 else 0 end      as OutcomeFinal
from    (
        select  t.*,
                case when t.CancelDate is null or t.CancelDate >= date_sub(t.RenewalDate, 30) then 1 else 0 end as DueFlag,
                case when m.cand_code is not null then 1 else 0 end                             as OpsRetained,
                m.cand_code                                                                     as OpsNewPolicyCode,
                m.cand_sale_date                                                                as OpsNewSaleDate,
                m.tier                                                                          as OpsMatchTier,
                case when t.PolicyRetNum = 1 or m.cand_code is not null then 1 else 0 end       as Renewed,
                cast(case when t.PolicyRetNum = 1 then t.TYReportingSaleDate else m.cand_sale_date end as date) as RenewedDate,
                case when t.PolicyRetNum = 1 then t.TYPolicyCode else m.cand_code end           as RenewedPolicyCode,
                case when t.PolicyRetNum = 1 then t.Channel else m.cand_channel end             as RenewedChannel,
                case when (case when t.PolicyRetNum = 1 then t.TYPolicyCode else m.cand_code end) = t.PolicyCode
                     then 1 else 0 end                                                          as RenewedSameCode,
                case when (case when t.PolicyRetNum = 1 then t.Channel else m.cand_channel end) = 'e) Online'
                       or o.PolicyCode is not null then 'online' else 'call centre' end         as RenewalMode
        from    ${ads.stg}.adsv2_renewal_terms t
        left join ${ads.stg}.adsv2_renewal_match m
                on m.product = t.LYPolicyTypeGroup and m.due_code = t.PolicyCode and m.due_date = t.RenewalDate
        left join (
                select  distinct b.PolicyCode, b.RenewalDate
                from    ${ads.stg}.adsv2_renewal_terms b
                join    ${ads.ods}.RenewalPortalEvent e on e.PolicyCode = b.PolicyCode
                where   e.EventType = 'PaymentSuccess'
                and     e.EventDateTime > date_sub(b.RenewalDate, 60)
                and     e.EventDateTime < date_add(b.RenewalDate, 40)
                ) o     on o.PolicyCode = t.PolicyCode and o.RenewalDate = t.RenewalDate
        ) x

## Shared: renewal_scv_quotes

Re-quotes (H8, D-E1.27): quotes by the renewing customer (SCV key, ods.scveadmlinkfinal) started between due - 60 and due + 30 days of the cycle, attached to that cycle only. 04 took every quote by the customer in 5 years (313,077 of 386,673 Motor quotes more than a year from any due date). Families keep the product's quote system.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_renewal_scv_quotes                                                          */
/*  Description : Renewal cycle x the same customer's quotes started between due - 60 and due + 30 days */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_renewal_base, ods.scveadmlinkfinal, ods_v2.Quote                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_renewal_scv_quotes as
select  distinct
        b.PolicyCode,
        b.RenewalDate,
        b.LYPolicyTypeGroup,
        q.SourceSystemId                                                as QuoteSourceSystemId,
        q.SourceSystemQuoteReference                                    as QuoteReference,
        q.QuoteId,
        q.QuoteDateTime
from    ${ads.stg}.adsv2_renewal_base b
join    ${ads.link}.scveadmlinkfinal l  on l.SCV_Customer_Key = b.SCV_Customer_Key and l.ChillSourceQuoteReference is not null
join    ${ads.ods}.Quote q              on q.SourceSystemId = l.SourceSystemId
                                       and q.SourceSystemQuoteReference = case when l.SourceSystemId = 2 then lower(trim(l.ChillSourceQuoteReference))
                                                                               else upper(trim(l.ChillSourceQuoteReference)) end
where   q.QuoteDateTime >= cast(date_sub(b.RenewalDate, 60) as timestamp)
and     q.QuoteDateTime <  cast(date_add(b.RenewalDate, 31) as timestamp)

## Shared: hfq_priced

HFQ price responses with a premium returned (04 cell 15, stg.hfq_prices); read by Home acquisition and renewal.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hfq_priced                                                                  */
/*  Description : HFQ price responses with a premium returned (Premium > 0, Outcome PremiumReturned).   */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteHomePrice                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hfq_priced as
select  QuoteCodeReference, QuoteId, DateCreated, TotalPayable
from    ${ads.ods}.QuoteHomePrice
where   Premium > 0
and     Outcome = 'PremiumReturned'

## Shared: hfq_payment

The latest HFQ payment response per payment reference (04 cell 33, stg.HFQ_Payments); read by Home acquisition and renewal.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_hfq_payment                                                                 */
/*  Description : The latest HFQ payment response per payment reference, with its quote (stg.HFQ_Paymen */
/*              : ts).                                                                                  */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteHomePaymentResponse                                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_hfq_payment as
select  QuoteReference, QuoteId, ActionResultCode, ActionTimeCreated
from    (
        select  QuoteReference, QuoteId, ActionResultCode, cast(ActionTimeCreated as timestamp) as ActionTimeCreated,
                row_number() over (partition by Reference order by cast(ActionTimeCreated as timestamp) desc) as rn
        from    ${ads.ods}.QuoteHomePaymentResponse
        where   Reference is not null
        ) r
where   rn = 1

## Shared: contact_link

The contact rule (design 08 §5.1; built here so families can attribute calls, published as ads_v2.ContactScvLink by ADS_v2_20_ScvAttach): email / phone hash keys of every source customer (ods_v2.SourceCustomer, no pii read) x the dated uses of that customer in ods.scveadmlinkfinal; at any time the SCV key with the latest use on or before it (ties: lowest key), with a 30-day look-ahead before a contact's first use. Also the customer's number per Genesys conversation.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_scv_link_use                                                                */
/*  Description : Every dated use of each scveadmlinkfinal source customer: quote start (QuoteStartDate */
/*              : , text in three formats), policy start, first / last sale and last renewal dates, and */
/*              :  for Relay clients (system 1) every record change of their policies (ods_v2.PolicyVer */
/*              : sion starts; the warehouse's telephony rule uses client or policy record changes, and */
/*              :  no client history is in ods_v2). Placeholder dates (1900 and before) are dropped.    */
/*              :                                                                                       */
/*              : Reads: ods.scveadmlinkfinal, ods_v2.PolicyVersion                                     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_scv_link_use as
select  u.RapierCustomerId,
        u.SCV_Customer_Key,
        u.SourceSystemId,
        u.UseSource,
        u.UseDateTime
from    (
        select  l.ChillSourceCustomerKey                                as RapierCustomerId,
                l.SCV_Customer_Key,
                l.SourceSystemId,
                s.UseSource,
                s.UseDateTime
        from    ${ads.link}.scveadmlinkfinal l
        lateral view stack(5,
                'quote start',  try_cast(replace(left(l.QuoteStartDate, 19), 'T', ' ') as timestamp),
                'policy start', cast(l.PolicyInceptDate as timestamp),
                'first sale',   cast(l.FirstSaleDate as timestamp),
                'last sale',    cast(l.LastSalesDate as timestamp),
                'last renewal', cast(l.LastRenewalDate as timestamp)) s as UseSource, UseDateTime
        union all
        select  l.ChillSourceCustomerKey,
                l.SCV_Customer_Key,
                l.SourceSystemId,
                'policy record change',
                v.RecordDateTimeStart
        from    (
                select  distinct ChillSourceCustomerKey, SCV_Customer_Key, SourceSystemId,
                        nullif(upper(trim(ChillSourcePolicyReference)), '')                as PolicyCode
                from    ${ads.link}.scveadmlinkfinal
                where   SourceSystemId = 1 and ChillSourcePolicyReference is not null
                ) l
        join    ${ads.ods}.PolicyVersion v  on v.PolicyCode = l.PolicyCode
        ) u
where   u.UseDateTime is not null
and     year(u.UseDateTime) > 1900
and     u.RapierCustomerId is not null

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_contact_use                                                                 */
/*  Description : Contact (EmailKey / PhoneKey from ods_v2.SourceCustomer) x use time: the lowest SCV k */
/*              : ey with a use at that time, how many SCV keys used it at that same time, and the use  */
/*              : source.                                                                               */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_scv_link_use, ods_v2.SourceCustomer                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_contact_use as
select  c.ContactKey,
        c.ContactKeyType,
        u.UseDateTime,
        min(u.SCV_Customer_Key)                                         as SCV_Customer_Key,
        count(distinct u.SCV_Customer_Key)                              as ScvCandidates,
        min_by(u.UseSource, u.SCV_Customer_Key)                         as UseSource
from    ${ads.stg}.adsv2_scv_link_use u
join    (
        select  RapierCustomerId, EmailKey as ContactKey, 'EmailKey' as ContactKeyType
        from    ${ads.ods}.SourceCustomer
        where   EmailKey is not null
        union all
        select  RapierCustomerId, PhoneKey, 'PhoneKey'
        from    ${ads.ods}.SourceCustomer
        where   PhoneKey is not null
        ) c     on c.RapierCustomerId = u.RapierCustomerId
group by c.ContactKey, c.ContactKeyType, u.UseDateTime

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_contact_island                                                              */
/*  Description : Consecutive uses of a contact by the same SCV key form one island (gaps and islands): */
/*              :  a new island starts whenever the SCV key with the latest use changes.                */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_contact_use                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_contact_island as
select  x.*,
        sum(x.IsChange) over (partition by x.ContactKey, x.ContactKeyType order by x.UseDateTime
                              rows between unbounded preceding and current row) as IslandNo
from    (
        select  u.*,
                case when lag(u.SCV_Customer_Key) over (partition by u.ContactKey, u.ContactKeyType
                                                        order by u.UseDateTime) <=> u.SCV_Customer_Key
                     then 0 else 1 end                                  as IsChange
        from    ${ads.stg}.adsv2_contact_use u
        ) x

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_contact_scv_link                                                            */
/*  Description : Email / phone hash key -> SCV key with a validity window [ValidFrom, ValidTo): the SC */
/*              : V key with the latest use on or before the time (ties: lowest key). One look-ahead wi */
/*              : ndow per contact covers the 30 days before its first use (a call before the first quo */
/*              : te); provisional when live (re-checked for 30 days). ContactScvKeyCount = how many SC */
/*              : V keys ever used the contact.                                                         */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_contact_island, stg.adsv2_contact_use                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_contact_scv_link as
select  ContactKey,
        ContactKeyType,
        SCV_Customer_Key,
        ValidFrom,
        ValidTo,
        LinkBasis,
        ScvCandidates,
        UseCount,
        FirstUseSource,
        size(collect_set(SCV_Customer_Key) over (partition by ContactKey, ContactKeyType)) as ContactScvKeyCount,
        '${ads.batch_id}'                                               as AdsBatchId
from    (
        select  ContactKey,
                ContactKeyType,
                SCV_Customer_Key,
                ValidFrom,
                lead(ValidFrom) over (partition by ContactKey, ContactKeyType order by ValidFrom) as ValidTo,
                'use'                                                   as LinkBasis,
                ScvCandidates,
                UseCount,
                FirstUseSource
        from    (
                select  ContactKey, ContactKeyType, IslandNo, SCV_Customer_Key,
                        min(UseDateTime)                                as ValidFrom,
                        min_by(ScvCandidates, UseDateTime)              as ScvCandidates,
                        count(*)                                        as UseCount,
                        min_by(UseSource, UseDateTime)                  as FirstUseSource
                from    ${ads.stg}.adsv2_contact_island
                group by ContactKey, ContactKeyType, IslandNo, SCV_Customer_Key
                ) i
        union all
        select  ContactKey,
                ContactKeyType,
                min_by(SCV_Customer_Key, UseDateTime),
                min(UseDateTime) - interval 30 days,
                min(UseDateTime),
                'look-ahead',
                min_by(ScvCandidates, UseDateTime),
                0,
                min_by(UseSource, UseDateTime)
        from    ${ads.stg}.adsv2_contact_use
        group by ContactKey, ContactKeyType
        ) w

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_call_contact                                                                */
/*  Description : The customer's number per Genesys conversation (voice and callback): the caller (ani) */
/*              :  on inbound conversations, the dialled number (dnis) on outbound ones, from the earli */
/*              : est session that has it. Replaces notebook 05's ods.customercallevents (raw phone num */
/*              : bers, built elsewhere).                                                               */
/*              :                                                                                       */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_call_contact as
select  ConversationId,
        min(ConversationStartTime)                                      as ConversationStartTime,
        min_by(PhoneKey, SessionStartTime)                              as PhoneKey,
        count(distinct PhoneKey)                                        as PhoneKeys
from    (
        select  ConversationId,
                ConversationStartTime,
                SessionStartTime,
                case when OriginatingDirection = 'outbound' then DnisPhoneKey else AniPhoneKey end as PhoneKey
        from    ${ads.ods}.CallSession
        where   ConversationStartTime >= add_months(timestamp '${ads.start_time}', -1) and ConversationStartTime < add_months(timestamp '${ads.end_time}', 1)
        ) c
where   PhoneKey is not null
group by ConversationId

## Shared: call_attribution

D-E1.24 for every journey: a call's customer is the SCV key holding its number at the call time (ContactScvLink); its policy is, for in-life calls, the customer's policy of the product open at the call date, and for renewal calls, the customer's renewal cycle open at the call date (due - 60 to due + 30 days, nearest due date). Replaces the interim phone -> Relay client -> month-end matches of the in-life and Van ports.


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_call_customer                                                               */
/*  Description : Each conversation's customer number and the SCV key holding it at the call start (nul */
/*              : l when none).                                                                         */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_call_contact, stg.adsv2_contact_scv_link                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_call_customer as
select  c.ConversationId,
        c.ConversationStartTime,
        c.PhoneKey,
        k.SCV_Customer_Key,
        k.LinkBasis
from    ${ads.stg}.adsv2_call_contact c
left join ${ads.stg}.adsv2_contact_scv_link k
        on  k.ContactKey = c.PhoneKey and k.ContactKeyType = 'PhoneKey'
        and c.ConversationStartTime >= k.ValidFrom
        and (k.ValidTo is null or c.ConversationStartTime < k.ValidTo)

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_call_open_policy                                                            */
/*  Description : Per conversation and product, the customer's Relay policies (scveadmlinkfinal system  */
/*              : 1) open at the call date (open at the month end before the call or at the call's own  */
/*              : month end, so a policy sold or cancelled in the month counts): how many, the lowest c */
/*              : ode and its id.                                                                       */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_call_customer, ods.scveadmlinkfinal, ods_v2.PolicyMonthEnd           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_call_open_policy as
select  c.ConversationId,
        m.PolicyTypeGroup,
        count(distinct m.PolicyCode)                                    as Policies,
        min(m.PolicyCode)                                               as PolicyCode,
        min_by(m.PolicyId, m.PolicyCode)                                as PolicyId
from    ${ads.stg}.adsv2_call_customer c
join    (
        select  distinct SCV_Customer_Key, nullif(upper(trim(ChillSourcePolicyReference)), '') as PolicyCode
        from    ${ads.link}.scveadmlinkfinal
        where   SourceSystemId = 1 and ChillSourcePolicyReference is not null
        ) l     on l.SCV_Customer_Key = c.SCV_Customer_Key
join    ${ads.ods}.PolicyMonthEnd m
        on  m.PolicyCode = l.PolicyCode
        and m.OpenNum = 1
        and cast(m.EffectiveDate as date) in (last_day(add_months(cast(c.ConversationStartTime as date), -1)),
                                              last_day(cast(c.ConversationStartTime as date)))
group by c.ConversationId, m.PolicyTypeGroup

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_call_renewal_policy                                                         */
/*  Description : Per conversation and product, the customer's renewal cycle open at the call date (due */
/*              :  - 60 to due + 30 days), the nearest due date first, then the lowest code (D-E1.24).  */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_call_customer, stg.adsv2_renewal_base                                */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_call_renewal_policy as
select  ConversationId, LYPolicyTypeGroup as PolicyTypeGroup, PolicyCode, RenewalDate, Cycles
from    (
        select  c.ConversationId,
                b.LYPolicyTypeGroup,
                b.PolicyCode,
                b.RenewalDate,
                count(*) over (partition by c.ConversationId, b.LYPolicyTypeGroup) as Cycles,
                row_number() over (partition by c.ConversationId, b.LYPolicyTypeGroup
                                   order by abs(datediff(cast(c.ConversationStartTime as date), b.RenewalDate)),
                                            b.PolicyCode) as rn
        from    ${ads.stg}.adsv2_call_customer c
        join    ${ads.stg}.adsv2_renewal_base b
                on  b.SCV_Customer_Key = c.SCV_Customer_Key
                and cast(c.ConversationStartTime as date) between date_sub(b.RenewalDate, 60) and date_add(b.RenewalDate, 30)
        ) x
where   rn = 1

## Shared: scv_lookups

Reference -> SCV key lookups from ods.scveadmlinkfinal (lowest key and how many), shared by the SCV attach (ADS_v2_20) and the campaign history (ADS_v2_23).


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_scv_by_policy                                                               */
/*  Description : Policy code -> lowest SCV key and how many (any source system holding the code, as 05 */
/*              :  cell 9).                                                                             */
/*              :                                                                                       */
/*              : Reads: ods.scveadmlinkfinal                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_scv_by_policy as
select  nullif(upper(trim(ChillSourcePolicyReference)), '')                                as PolicyRef,
        min(SCV_Customer_Key)                                           as SCV_Customer_Key,
        count(distinct SCV_Customer_Key)                                as ScvCandidates
from    ${ads.link}.scveadmlinkfinal
where   ChillSourcePolicyReference is not null
group by nullif(upper(trim(ChillSourcePolicyReference)), '')

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_scv_by_quote                                                                */
/*  Description : Quote reference (lower) x source system -> lowest SCV key and how many.               */
/*              :                                                                                       */
/*              : Reads: ods.scveadmlinkfinal                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_scv_by_quote as
select  nullif(lower(trim(ChillSourceQuoteReference)), '')                                 as QuoteRef,
        SourceSystemId,
        min(SCV_Customer_Key)                                           as SCV_Customer_Key,
        count(distinct SCV_Customer_Key)                                as ScvCandidates
from    ${ads.link}.scveadmlinkfinal
where   ChillSourceQuoteReference is not null
group by nullif(lower(trim(ChillSourceQuoteReference)), ''), SourceSystemId

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_scv_by_customer                                                             */
/*  Description : Customer reference -> lowest SCV key and how many, for Relay clients (system 1, clien */
/*              : t code) and Mapfre travel customers (system 5, 'M' + customer id).                    */
/*              :                                                                                       */
/*              : Reads: ods.scveadmlinkfinal                                                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_scv_by_customer as
select  SourceSystemId,
        nullif(upper(trim(ChillSourceCustomerReference)), '')                              as CustomerRef,
        min(SCV_Customer_Key)                                           as SCV_Customer_Key,
        count(distinct SCV_Customer_Key)                                as ScvCandidates
from    ${ads.link}.scveadmlinkfinal
where   SourceSystemId in (1, 5) and ChillSourceCustomerReference is not null
group by SourceSystemId, nullif(upper(trim(ChillSourceCustomerReference)), '')

In [ ]:
%%pyspark
ads_run_log("10_Common", "succeeded")
